## Cerebellar grey matter volume extractor

Using SUITPy

In [ ]:
# Import packages
from nilearn import plotting as npl
import SUITPy as suit
import SUITPy.atlas as atlas
import nibabel as nib
import ants
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path # for saving file name

In [ ]:
"""
Modified SUITPy.normalization function: added args jac_log = False, jac_geom = True
Make sure to also add the template (in folder "templates")
"""
import sys
sys.path.append('/Users/medha/Desktop/USRI 2026/suitpy edits')

from normalization import normalize # import from above folder

import SUITPy.normalization as modif

modif.normalize = normalize # SUITPy.normalization normalize function now uses local (edited) normalize function

In [ ]:
t1_img = 
gm_img = 
file_name = Path(t1_img).stem

In [ ]:
# new folder for each individual's outputs
new_path = Path('gmv_results') / file_name
new_path.mkdir(parents = True)

In [ ]:
# 1. isolation module

"""
`isolate` function generates an isolation MASK for cerebellum; registration done by function
input: T1-weighted scan
output: cerebellar mask (as ANTsImage)
"""
mask = suit.isolate(t1_img, result_folder = new_path)
mask_path = new_path/f'{file_name}_cerebellum_dseg.nii.gz'

"""
# visualize results
img = nib.load(t1_img)
mask = nib.load(str_mask_path)
npl.plot_roi(mask_img, img)
"""

In [ ]:
# 2. normalization module

"""
Input: cerebellar mask (from isolation); source image
Output: (1) normalized cerebellum: anatomical in SUIT space; (2) deformation field
Normalizes to the SUIT cerebellar template
"""

results = modif.normalize(source_file = t1_img,
                         mask_file = str(mask_path),
                         space = 'SUIT', # SUIT is default
                         write_jacobian_determinant=True, # save jac det
                         write_ants_transform = True, # forward and inverse ANTs transforms
                         write_normalized = True, # save template space
                         result_folder = new_path,
                         jac_log = False,
                         jac_geom = True, # robust to different image orientations (e.g. non-axial)
                         verbose = 1)

In [ ]:
"""
Reslice module:
`reslice_image` deforms images aligned to indiv anat img into SUIT space
So use the isolation mask (isolation) and deformation file (normalization)
"""

In [ ]:
# 3. reslice image

"""
Input: cerebellar gm (c1), deformation image, cerebellar mask
Output: resliced image
"""
# (anatomical) grey matter in suit space
output_img = suit.reslice_image(source_image = gm_img , # want grey matter mapped from naive space to suit space
                                deformation = results['fwd_deformation'],
                                mask = str(mask_path) # nifti, str
)

nib.save(output_img, new_path/f'output_{file_name}.nii')

# optional plotting
#npl.plot_anat(output_img, cut_coords=(-1, -58, -36)) # specify 3-dim coordinates


In [ ]:
npl.plot_anat(output_img)

In [ ]:
# calculate gmv in suit space

mask_img = nib.load(str(mask_path))
voxel_dim = mask_img.header.get_zooms()[:3] # voxel volume DIMENSIONS, also check in fsleyes
voxel_vol = np.prod(voxel_dim) # volume of each voxel

# mask GM with isolated cerebellar mask
gm_data = output_img.get_fdata() # grey matter in suit space
jac_det_img = nib.load(results['jacobian_determinant']) # nifti
jac_det = jac_det_img.get_fdata() # numpy array

gmv = gm_data*jac_det*voxel_vol

In [ ]:
gmv_img = nib.Nifti1Image(gmv, output_img.affine)
nib.save(gmv_img, new_path/f'gmv_img_{file_name}.nii')

In [ ]:
voxel_dim_gmv = gmv_img.header.get_zooms()[:3] # voxel volume DIMENSIONS, also check in fsleyes
voxel_vol_gmv = np.prod(voxel_dim) # volume of each voxel
# should be same as mask

In [ ]:
# 4. dataframe for resliced image (choose an atlas)

# in this case, using Nettekoven_2024 with asymmetric 32-region map (for hand function region, M3)
atlas.fetch_atlas('Nettekoven_2024')
df = atlas.summarize_data(gmv_img, space = 'SUIT', # use same space as normalization
                     stats = ['nanmean'],
                     atlas = 'Nettekoven_2024', maps = 'atl-NettekovenAsym32')
df['gmv'] = df['nanmean']*(df['size']/voxel_vol_gmv) # total volume = sum of gmv/voxel
 # to get the sum: size is volume of region (mm^3, voxel), so number of voxels is size / voxel volume

#df.rename(columns={'nanmean': 'GM_prob'}, inplace = True)

df['image_name'] = f'gmv_img_{file_name}.nii' # file name for each
df.drop(columns = 'nanmean', inplace = True)
df.head(5)

In [ ]:
df.to_csv('gmv_results/gmv_atlas.csv', mode = 'a', index = True, header = False) # header only for first run